In [1]:
import logging
import time
import os
import pickle

#from DataPrep import data_prep

import numpy as np
import matplotlib.pyplot as plt

#import tensorflow_datasets as tfds
import tensorflow as tf

# Import tf_text to load the ops used by the tokenizer saved model
#import tensorflow_text  # pylint: disable=unused-import
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib as plt


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model,  Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dropout, Input, TimeDistributed, Dense, Activation, RepeatVector, Embedding, Concatenate
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Attention
from tensorflow.keras.optimizers import Adam, Adagrad
from keras.losses import sparse_categorical_crossentropy
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings
import random

In [2]:
with open("Pichia_TrTs_2Target.pkl", "rb") as fp:
    Data_AllOrg = pickle.load(fp)
    
AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']

In [3]:
Settings = pd.read_csv('../BO_forHyperParameter/Arch1/InitialRound_HyperParameter.csv').iloc[:, 1:]
Settings

,Enc hidden size,Enc Embedding size,Dec Embedding size,Dense Layer size,Dense Layer size aa,Drop rate,Drop rate aa
0,410.0,164.0,85.0,166.0,104.0,0.1,0.5
1,203.0,54.0,60.0,146.0,54.0,0.0,0.8
2,453.0,236.0,229.0,186.0,110.0,0.6,0.6
3,367.0,193.0,190.0,238.0,30.0,0.9,0.1
4,63.0,177.0,208.0,250.0,163.0,0.2,0.6
5,122.0,109.0,155.0,208.0,183.0,0.6,0.9
6,288.0,39.0,93.0,92.0,133.0,0.5,0.2
7,244.0,209.0,168.0,67.0,189.0,0.8,0.4
8,267.0,83.0,121.0,128.0,232.0,0.4,0.3
9,166.0,95.0,137.0,111.0,208.0,0.7,0.7


#### Network parameters

In [4]:
for i in np.arange(10, 12):
    Setting_no = i

    Max_length = 1000
    learning_rate = 0.001
    batch_size = 150
    epochs = 100
    aa_vocab_size = 25
    dna_vocab_size = 67


    hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
    hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
    embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
    embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
    Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
    Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])

    drop_rate = Settings['Drop rate'][Setting_no]
    drop_rate_aa = Settings['Drop rate aa'][Setting_no]

    input_sequence = Input(shape=(Max_length,))
    encod_emb = Embedding(input_dim= aa_vocab_size, output_dim = embedding_size_enc,trainable=True, mask_zero = True)
    embedding = encod_emb(input_sequence)

    encoder = Bidirectional(GRU(hidden_size_enc, return_sequences=True, return_state = True),
                            merge_mode="concat", weights=None)

    encoder_sequence, encoder_final_f, encoder_final_b  = encoder(embedding)

    encoder_final = Concatenate(axis=-1)([encoder_final_f, encoder_final_b])


    
    decoder_inputs = Input(shape=(Max_length -1, ))
    decoder_inputs_aa = Input(shape=(Max_length, ))

    dex=  Embedding(input_dim = dna_vocab_size, output_dim = embedding_size_dec, trainable=True, mask_zero = True)


    final_dex= dex(decoder_inputs)
    final_dex_aa =  encod_emb(decoder_inputs_aa)


    decoder = GRU(2*hidden_size_enc, return_sequences = True, return_state = True)
    decoder_aa =  GRU(2*hidden_size_enc_aa, return_sequences = True, return_state = True)

    decoder_sequence, decoder_final = decoder(final_dex, initial_state=encoder_final)
    decoder_sequence_aa, decoder_final_aa = decoder_aa(final_dex_aa, initial_state=encoder_final)


    attn_layer = Attention()
    attn_out = attn_layer([decoder_sequence, encoder_sequence])
    attn_out_aa = attn_layer([decoder_sequence_aa, encoder_sequence])

    decoder_concat_input = Concatenate(axis=-1)([decoder_sequence, attn_out]) #decoder_sequence, 
    decoder_concat_input_aa = Concatenate(axis=-1)([decoder_sequence_aa, attn_out_aa]) #decoder_sequence,


    Intermediate_layer = TimeDistributed(Dense(Dense_layer_size, activation='tanh'))
    Intermediate_layer_aa= TimeDistributed(Dense(Dense_layer_size_aa, activation='tanh'))

    Intemediate_output = Intermediate_layer(decoder_concat_input) #decoder_concat_input
    Intemediate_output_aa = Intermediate_layer_aa(decoder_concat_input_aa) #decoder_concat_input


    dropout_layer = Dropout(drop_rate)
    dropout_output = dropout_layer(Intemediate_output)

    dropout_layer_aa = Dropout(drop_rate_aa)
    dropout_output_aa = dropout_layer(Intemediate_output_aa)

    dense_layer = TimeDistributed(Dense(dna_vocab_size, activation='softmax'))
    logits = dense_layer(dropout_output)

    dense_layer_aa = TimeDistributed(Dense(aa_vocab_size, activation='softmax'))
    logits_aa = dense_layer_aa(dropout_output_aa)

    enc_dec_model = Model([input_sequence, decoder_inputs, decoder_inputs_aa], [logits, logits_aa])

    enc_dec_model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate = learning_rate),
                  metrics=['accuracy'])
    enc_dec_model.summary()
    
    checkpoint_path = "/Desktop/training_2Target/Initial/" + str(Setting_no) + "/cp.ckpt"
    checkpoint_dir = os.path.dirname(checkpoint_path)

    # Create a callback that saves the model's weights
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)

    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience = 3,
        verbose=0, mode="auto", baseline=None, restore_best_weights=False)
    ## Train the model
    model_results = enc_dec_model.fit([AA_tr[:,1:Max_length+1], Cds_tr[:,0:Max_length-1], AA_tr[:,0:Max_length]], 
                                      [Cds_tr[:,1:Max_length],  AA_tr[:,1:Max_length+1]], 
                                      batch_size= batch_size, 
                                      epochs= epochs, 
                                  validation_split=0.2, callbacks=[cp_callback, early_stop])
    
    name_history = 'Loss_Evolution/Initial/Combo'+ str(Setting_no) + '.csv'
    pd.DataFrame(model_results.history).to_csv(name_history)
    
    name_model = 'saved_models/Initial/EncDec_' + str(Setting_no)
    enc_dec_model.save(name_model)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, 1000, 249)    6225        ['input_1[0][0]',                
                                                                  'input_3[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional (Bidirectional)  [(None, 1000, 720),  1319760     ['embedding[0][0]']          

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4707 - time_distributed_2_loss: 0.4696 - time_distributed_3_loss: 0.0011 - time_distributed_2_accuracy: 0.4761 - time_distributed_3_accuracy: 0.9997
Epoch 5: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt
133/133 [==============================] - 95s 712ms/step - loss: 0.4707 - time_distributed_2_loss: 0.4696 - time_distributed_3_loss: 0.0011 - time_distributed_2_accuracy: 0.4761 - time_distributed_3_accuracy: 0.9997 - val_loss: 0.4613 - val_time_distributed_2_loss: 0.4604 - val_time_distributed_3_loss: 8.9278e-04 - val_time_distributed_2_accuracy: 0.4852 - val_time_distributed_3_accuracy: 0.9997
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4649 - time_distributed_2_loss: 0.4640 - time_distributed_3_loss: 9.4066e-04 - time_distributed_2_accuracy: 0.4830 - time_distributed_3_accuracy: 0.9997
Epoch 6: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt
133/133 [===

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.4469 - time_distributed_2_loss: 0.4465 - time_distributed_3_loss: 4.1303e-04 - time_distributed_2_accuracy: 0.5059 - time_distributed_3_accuracy: 0.9998
Epoch 17: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt
133/133 [==============================] - 92s 689ms/step - loss: 0.4469 - time_distributed_2_loss: 0.4465 - time_distributed_3_loss: 4.1303e-04 - time_distributed_2_accuracy: 0.5059 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4465 - val_time_distributed_2_loss: 0.4459 - val_time_distributed_3_loss: 5.9221e-04 - val_time_distributed_2_accuracy: 0.5058 - val_time_distributed_3_accuracy: 0.9998
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.4451 - time_distributed_2_loss: 0.4447 - time_distributed_3_loss: 3.9422e-04 - time_distributed_2_accuracy: 0.5089 - time_distributed_3_accuracy: 0.9998
Epoch 18: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt


Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.3701 - time_distributed_2_loss: 0.3697 - time_distributed_3_loss: 4.2983e-04 - time_distributed_2_accuracy: 0.6309 - time_distributed_3_accuracy: 0.9998
Epoch 29: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt
133/133 [==============================] - 86s 646ms/step - loss: 0.3701 - time_distributed_2_loss: 0.3697 - time_distributed_3_loss: 4.2983e-04 - time_distributed_2_accuracy: 0.6309 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.3908 - val_time_distributed_2_loss: 0.3904 - val_time_distributed_3_loss: 4.5369e-04 - val_time_distributed_2_accuracy: 0.6065 - val_time_distributed_3_accuracy: 0.9998
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.3593 - time_distributed_2_loss: 0.3589 - time_distributed_3_loss: 3.8776e-04 - time_distributed_2_accuracy: 0.6460 - time_distributed_3_accuracy: 0.9998
Epoch 30: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt


Epoch 41/100
133/133 [==============================] - ETA: 0s - loss: 0.2800 - time_distributed_2_loss: 0.2796 - time_distributed_3_loss: 3.5356e-04 - time_distributed_2_accuracy: 0.7422 - time_distributed_3_accuracy: 0.9999
Epoch 41: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt
133/133 [==============================] - 84s 630ms/step - loss: 0.2800 - time_distributed_2_loss: 0.2796 - time_distributed_3_loss: 3.5356e-04 - time_distributed_2_accuracy: 0.7422 - time_distributed_3_accuracy: 0.9999 - val_loss: 0.3336 - val_time_distributed_2_loss: 0.3331 - val_time_distributed_3_loss: 5.2920e-04 - val_time_distributed_2_accuracy: 0.7009 - val_time_distributed_3_accuracy: 0.9998
Epoch 42/100
133/133 [==============================] - ETA: 0s - loss: 0.2765 - time_distributed_2_loss: 0.2760 - time_distributed_3_loss: 5.2436e-04 - time_distributed_2_accuracy: 0.7463 - time_distributed_3_accuracy: 0.9998
Epoch 42: saving model to /Desktop/training_2Target/Initial/10\cp.ckpt


...

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4563 - time_distributed_6_loss: 0.4558 - time_distributed_7_loss: 5.3691e-04 - time_distributed_6_accuracy: 0.4892 - time_distributed_7_accuracy: 0.9998
Epoch 5: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt
133/133 [==============================] - 122s 921ms/step - loss: 0.4563 - time_distributed_6_loss: 0.4558 - time_distributed_7_loss: 5.3691e-04 - time_distributed_6_accuracy: 0.4892 - time_distributed_7_accuracy: 0.9998 - val_loss: 0.4518 - val_time_distributed_6_loss: 0.4514 - val_time_distributed_7_loss: 4.3379e-04 - val_time_distributed_6_accuracy: 0.4940 - val_time_distributed_7_accuracy: 0.9998
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4540 - time_distributed_6_loss: 0.4536 - time_distributed_7_loss: 4.4297e-04 - time_distributed_6_accuracy: 0.4913 - time_distributed_7_accuracy: 0.9998
Epoch 6: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt
133

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.4413 - time_distributed_6_loss: 0.4410 - time_distributed_7_loss: 3.5383e-04 - time_distributed_6_accuracy: 0.5133 - time_distributed_7_accuracy: 0.9998
Epoch 17: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt
133/133 [==============================] - 124s 930ms/step - loss: 0.4413 - time_distributed_6_loss: 0.4410 - time_distributed_7_loss: 3.5383e-04 - time_distributed_6_accuracy: 0.5133 - time_distributed_7_accuracy: 0.9998 - val_loss: 0.4411 - val_time_distributed_6_loss: 0.4406 - val_time_distributed_7_loss: 4.9915e-04 - val_time_distributed_6_accuracy: 0.5133 - val_time_distributed_7_accuracy: 0.9998
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.4386 - time_distributed_6_loss: 0.4381 - time_distributed_7_loss: 4.6192e-04 - time_distributed_6_accuracy: 0.5189 - time_distributed_7_accuracy: 0.9998
Epoch 18: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt

Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.3225 - time_distributed_6_loss: 0.3221 - time_distributed_7_loss: 3.9081e-04 - time_distributed_6_accuracy: 0.6934 - time_distributed_7_accuracy: 0.9998
Epoch 29: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt
133/133 [==============================] - 123s 922ms/step - loss: 0.3225 - time_distributed_6_loss: 0.3221 - time_distributed_7_loss: 3.9081e-04 - time_distributed_6_accuracy: 0.6934 - time_distributed_7_accuracy: 0.9998 - val_loss: 0.3520 - val_time_distributed_6_loss: 0.3515 - val_time_distributed_7_loss: 4.6439e-04 - val_time_distributed_6_accuracy: 0.6628 - val_time_distributed_7_accuracy: 0.9998
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.3050 - time_distributed_6_loss: 0.3045 - time_distributed_7_loss: 4.6121e-04 - time_distributed_6_accuracy: 0.7147 - time_distributed_7_accuracy: 0.9998
Epoch 30: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt

Epoch 41/100
133/133 [==============================] - ETA: 0s - loss: 0.1750 - time_distributed_6_loss: 0.1746 - time_distributed_7_loss: 4.1638e-04 - time_distributed_6_accuracy: 0.8552 - time_distributed_7_accuracy: 0.9998
Epoch 41: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt
133/133 [==============================] - 123s 923ms/step - loss: 0.1750 - time_distributed_6_loss: 0.1746 - time_distributed_7_loss: 4.1638e-04 - time_distributed_6_accuracy: 0.8552 - time_distributed_7_accuracy: 0.9998 - val_loss: 0.2618 - val_time_distributed_6_loss: 0.2613 - val_time_distributed_7_loss: 4.7845e-04 - val_time_distributed_6_accuracy: 0.8023 - val_time_distributed_7_accuracy: 0.9998
Epoch 42/100
133/133 [==============================] - ETA: 0s - loss: 0.1685 - time_distributed_6_loss: 0.1681 - time_distributed_7_loss: 4.3032e-04 - time_distributed_6_accuracy: 0.8616 - time_distributed_7_accuracy: 0.9998
Epoch 42: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt

Epoch 53/100
133/133 [==============================] - ETA: 0s - loss: 0.1243 - time_distributed_6_loss: 0.1239 - time_distributed_7_loss: 3.9848e-04 - time_distributed_6_accuracy: 0.9020 - time_distributed_7_accuracy: 0.9999
Epoch 53: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt
133/133 [==============================] - 125s 943ms/step - loss: 0.1243 - time_distributed_6_loss: 0.1239 - time_distributed_7_loss: 3.9848e-04 - time_distributed_6_accuracy: 0.9020 - time_distributed_7_accuracy: 0.9999 - val_loss: 0.2478 - val_time_distributed_6_loss: 0.2473 - val_time_distributed_7_loss: 5.2532e-04 - val_time_distributed_6_accuracy: 0.8396 - val_time_distributed_7_accuracy: 0.9998
Epoch 54/100
133/133 [==============================] - ETA: 0s - loss: 0.1236 - time_distributed_6_loss: 0.1232 - time_distributed_7_loss: 3.6765e-04 - time_distributed_6_accuracy: 0.9024 - time_distributed_7_accuracy: 0.9999
Epoch 54: saving model to /Desktop/training_2Target/Initial/11\cp.ckpt

In [ ]:
name_history = 'Loss_Evolution/Initial/Combo'+ str(Setting_no) + '.csv'
pd.DataFrame(model_results.history).to_csv(name_history)

name_model = 'saved_models/Initial/EncDec_' + str(Setting_no)
enc_dec_model.save(name_model)

In [ ]:
name_model = 'saved_models/Initial/EncDec_' + str(Setting_no)
enc_dec_model.save(name_model)

#### Length and its plot

In [5]:
length_aa = []
for i in range(len(AA_seq_tokenized)):
    length_aa.append(len(AA_seq_tokenized[i]))
print(max(length_aa))

sns.histplot(length_aa)


NameError: name 'AA_seq_tokenized' is not defined

In [ ]:
 Cds_ts[0,0]

In [ ]:
Setting_no